# Forecasting time series with skforecast-ai

## Introduction

## Libraries

Libraries used in this document.

In [8]:
# Data processing
# ==============================================================================
import numpy as np
import pandas as pd
from rich.console import Console
from rich.panel import Panel
from rich.syntax import Syntax
from rich.text import Text
from skforecast.datasets import fetch_dataset

# Plots
# ==============================================================================
from skforecast.plot import set_dark_theme
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.io as pio
import plotly.offline as poff
pio.templates.default = "seaborn"
poff.init_notebook_mode(connected=True)
plt.style.use('seaborn-v0_8-darkgrid')

# Modelling and Forecasting
# ==============================================================================
import skforecast
import skforecast_ai
import lightgbm
from skforecast_ai import ForecastingAssistant
from skforecast.model_selection import TimeSeriesFold

# Warnings configuration
# ==============================================================================
import warnings
warnings.filterwarnings('once')

color = '\033[1m\033[38;5;208m' 
print(f"{color}Version skforecast_ai: {skforecast_ai.__version__}")
print(f"{color}Version skforecast: {skforecast.__version__}")
print(f"{color}Version lightgbm: {lightgbm.__version__}")

Version skforecast_ai: 0.1.0
Version skforecast: 0.23.0
Version lightgbm: 4.6.0


## Data

The data in this document represent the hourly usage of the bike share system in the city of Washington, D.C. during the years 2011 and 2012. In addition to the number of users per hour, information about weather conditions and holidays is available. The original data was obtained from the [UCI Machine Learning Repository](https://archive.ics.uci.edu/ml/datasets/bike+sharing+dataset) and has been previously cleaned ([code](https://github.com/JoaquinAmatRodrigo/Estadistica-machine-learning-python/blob/master/code/prepare_bike_sharing_dataset.ipynb)) applying the following modifications:

+ Renamed columns with more descriptive names.

+ Renamed categories of the weather variables. The category of `heavy_rain` has been combined with that of `rain`.

+ Denormalized temperature, humidity and wind variables.

+ `date_time` variable created and set as index.

+ Imputed missing values by forward filling.

In [9]:
# Downloading data
# ==============================================================================
data = fetch_dataset('bike_sharing', raw=True)
data = data[['date_time', 'users', 'holiday', 'weather', 'temp']]
data['date_time'] = pd.to_datetime(data['date_time'], format='%Y-%m-%d %H:%M:%S')
data = data.set_index('date_time')
data = data.asfreq('h')
data = data.sort_index()
data.head()

╭───────────────────────────────── bike_sharing ──────────────────────────────────╮
│ Description:                                                                    │
│ Hourly usage of the bike share system in the city of Washington D.C. during the │
│ years 2011 and 2012. In addition to the number of users per hour, information   │
│ about weather conditions and holidays is available.                             │
│                                                                                 │
│ Source:                                                                         │
│ Fanaee-T,Hadi. (2013). Bike Sharing Dataset. UCI Machine Learning Repository.   │
│ https://doi.org/10.24432/C5W894.                                                │
│                                                                                 │
│ URL:                                                                            │
│ https://raw.githubusercontent.com/skforecast/skforecast-                        │
│ datasets/main/data/bike_sharing_dataset_clean.csv                               │
│                                                                                 │
│ Shape: 17544 rows x 12 columns                                                  │
╰─────────────────────────────────────────────────────────────────────────────────╯

,users,holiday,weather,temp
date_time,,,,
2011-01-01 00:00:00,16.0,0.0,clear,9.84
2011-01-01 01:00:00,40.0,0.0,clear,9.02
2011-01-01 02:00:00,32.0,0.0,clear,9.02
2011-01-01 03:00:00,13.0,0.0,clear,9.84
2011-01-01 04:00:00,1.0,0.0,clear,9.84


In [10]:
assistant = ForecastingAssistant()

In [11]:
console = Console()
profile = assistant.profile(
             data        = data,
             target      = "users",
             date_column = "date_time",
         )
console.print(profile.model_dump_json(indent=2))

{
  "data_profile": {
    "data_format": "single",
    "n_series": 1,
    "series_lengths": {
      "users": {
        "start": "2011-01-01",
        "end": "2012-12-31 23:00:00",
        "length": 17544
      }
    },
    "span_index_length": 17544,
    "n_total_observations": 17544,
    "target": "users",
    "target_dtype": "numeric",
    "target_stats": {
      "users": {
        "min": 1.0,
        "max": 977.0,
        "mean": 187.79508663930687,
        "std": 181.34524235261304
      }
    },
    "missing_target": {},
    "date_column": null,
    "series_id_column": null,
    "index_type": "datetime",
    "frequency": "h",
    "frequency_is_set": true,
    "index_is_monotonic": true,
    "has_gaps": false,
    "has_duplicate_timestamps": false,
    "exog_columns": [
      "holiday",
      "weather",
      "temp"
    ],
    "categorical_exog": [
      "weather"
    ],
    "missing_exog": {},
    "data_path": "data.csv",
    "start_date": "2011-01-01",
    "end_train": "2012-08-07 18:00:00",
    "warnings": []
  },
  "task_type": "single_series",
  "forecaster": "ForecasterRecursive",
  "forecaster_candidates": [
    "ForecasterRecursive",
    "ForecasterDirect",
    "ForecasterFoundation",
    "ForecasterStats"
  ],
  "estimator": "LGBMRegressor",
  "estimator_candidates": [
    "LGBMRegressor",
    "XGBRegressor",
    "Ridge"
  ],
  "series_pacf": [
    {
      "series_id": "users",
      "n_observations": 17544,
      "lags": [
        1,
        2,
        23,
        22,
        25,
        169,
        10,
        145,
        17,
        143,
        167,
        19,
        21,
        337,
        3,
        20,
        142,
        32,
        26,
        135,
        166,
        8,
        160,
        15,
        121,
        5,
        119,
        136,
        313,
        335,
        24,
        33
      ],
      "pacf_abs": [
        0.8451720156882963,
        0.4081855899587966,
        0.3544857712328617,
        0.34267336940457616,
        0.33189603349204894,
        0.3152610898864623,
        0.27231484396124345,
        0.2436596981295079,
        0.24152942731810653,
        0.24031390765801355,
        0.2175130026791719,
        0.1988250786151921,
        0.1930441021006376,
        0.18414175330540225,
        0.18188835641757217,
        0.1781335380099876,
        0.15492433010110218,
        0.1533892188911791,
        0.14580001154579156,
        0.13605567975003616,
        0.1320985809065835,
        0.1318101704685699,
        0.1205043952301111,
        0.12040631617682111,
        0.11938436228876852,
        0.11782596097465975,
        0.11730947832466825,
        0.11655985116216161,
        0.11519854958525054,
        0.11272186409748103,
        0.11253526037526702,
        0.10068469228351072
      ]
    }
  ],
  "window_features": [
    {
      "stats": [
        "mean",
        "std"
      ],
      "window_sizes": 3
    },
    {
      "stats": [
        "mean"
      ],
      "window_sizes": 24
    },
    {
      "stats": [
        "mean"
      ],
      "window_sizes": 168
    }
  ],
  "calendar_features": [
    "hour",
    "day_of_week",
    "weekend",
    "month"
  ],
  "explanation": "A single-series ML forecaster (ForecasterRecursive) is recommended. Data: 17544 observations, 'h'
frequency. Alternative forecasters: ['ForecasterDirect', 'ForecasterFoundation', 'ForecasterStats']. Estimator: 
LGBMRegressor. A gradient boosting model is preferred for a dataset of this size (17544 observations). Alternative 
estimators: ['XGBRegressor', 'Ridge']. 3 exogenous variables (1 categorical) available as predictors."
}

In [12]:
result = assistant.forecast(
             data        = data,
             target      = "users",
             date_column = "date_time",
             steps       = 36,
         )

In [13]:
# Forecasted values for the next 12 steps
display(result.predictions.head(3))

# Evaluation metrics: MAE, MSE, MASE per series
display(result.metrics)

,pred
2012-08-07 19:00:00,602.275744
2012-08-07 20:00:00,460.290671
2012-08-07 21:00:00,326.293737


,series,MAE,MSE,MASE,MAPE
0,users,21.750571,774.745201,0.366468,0.175919


In [14]:
syntax = Syntax(
    result.code, 
    "python", 
    theme="monokai",
    word_wrap=True
)
console = Console()
console.print(syntax)

import pandas as pd                                                                                                
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error                
from skforecast.metrics import mean_absolute_scaled_error                                                          
from lightgbm import LGBMRegressor                                                                                 
from skforecast.preprocessing import RollingFeatures, CalendarFeatures                                             
from skforecast.recursive import ForecasterRecursive                                                               
                                                                                                                   
# Load data                                                                                                        
data = pd.read_csv('data.csv', index_col=0, parse_dates=True)                                                      
                                                                                                                   
data = data.asfreq('h')                                                                                            
data = data.sort_index()                                                                                           
                                                                                                                   
# Train/test split                                                                                                 
end_train = '2012-08-07 18:00:00'  # 80% of data, adjust to change the split point                                 
data_train = data.loc[:end_train]                                                                                  
data_test  = data.loc[data.index > end_train]                                                                      
exog_features = ['holiday', 'weather', 'temp']                                                                     
                                                                                                                   
print(                                                                                                             
    f"Train dates : {data_train.index.min()} --- {data_train.index.max()}  (n={len(data_train)})"                  
)                                                                                                                  
print(                                                                                                             
    f"Test dates  : {data_test.index.min()} --- {data_test.index.max()}  (n={len(data_test)})"                     
)                                                                                                                  
                                                                                                                   
window_features = RollingFeatures(                                                                                 
    stats        = ['mean', 'std', 'mean', 'mean'],                                                                
    window_sizes = [3, 3, 24, 168],                                                                                
)                                                                                                                  
                                                                                                                   
calendar_features = CalendarFeatures(                                                                              
    features = ['hour', 'day_of_week', 'weekend', 'month'],                                                        
[38;2;2

In [15]:
# Split train-validation-test
# ==============================================================================
end_train = '2012-04-30 23:59:00'
end_validation = '2012-08-31 23:59:00'
data_train = data.loc[: end_train, :]
data_val   = data.loc[end_train:end_validation, :]
data_test  = data.loc[end_validation:, :]

print(f"Dates train      : {data_train.index.min()} --- {data_train.index.max()}  (n={len(data_train)})")
print(f"Dates validation : {data_val.index.min()} --- {data_val.index.max()}  (n={len(data_val)})")
print(f"Dates test       : {data_test.index.min()} --- {data_test.index.max()}  (n={len(data_test)})")

Dates train      : 2011-01-01 00:00:00 --- 2012-04-30 23:00:00  (n=11664)
Dates validation : 2012-05-01 00:00:00 --- 2012-08-31 23:00:00  (n=2952)
Dates test       : 2012-09-01 00:00:00 --- 2012-12-31 23:00:00  (n=2928)


In [16]:
cv = TimeSeriesFold(steps = 36, initial_train_size = end_validation)
result = assistant.backtest(
             data        = data,
             target      = "users",
             date_column = "date_time",
             cv          = cv,
         )

  0%|          | 0/82 [00:00<?, ?it/s]

Information of folds
--------------------
Number of observations used for initial training: 14616
Number of observations used for backtesting: 2928
    Number of folds: 82
    Number skipped folds: 0 
    Number of steps per fold: 36
    Number of steps to exclude between last observed data (last window) and predictions (gap): 0
    Last fold only includes 12 observations.

Fold: 0
    Training:   2011-01-01 00:00:00 -- 2012-08-31 23:00:00  (n=14616)
    Validation: 2012-09-01 00:00:00 -- 2012-09-02 11:00:00  (n=36)
Fold: 1
    Training:   No training in this fold
    Validation: 2012-09-02 12:00:00 -- 2012-09-03 23:00:00  (n=36)
Fold: 2
    Training:   No training in this fold
    Validation: 2012-09-04 00:00:00 -- 2012-09-05 11:00:00  (n=36)
Fold: 3
    Training:   No training in this fold
    Validation: 2012-09-05 12:00:00 -- 2012-09-06 23:00:00  (n=36)
Fold: 4
    Training:   No training in this fold
    Validation: 2012-09-07 00:00:00 -- 2012-09-08 11:00:00  (n=36)
Fold: 5
    Tr

100%|██████████| 82/82 [00:01<00:00, 52.94it/s]


In [17]:
display(result.metrics)
print(result.cv_config) # exactly which folds were run

console = Console()
panel = Panel(
        Text(result.explanation, justify="left"),
        title        = "Skforecast-AI Results Explanation",
        title_align  = "center",
        border_style = "color(214)",
        width        = 88,
    )
console.print(panel)

,mean_absolute_error,mean_squared_error,mean_absolute_scaled_error,mean_absolute_percentage_error
0,46.584343,5443.171132,0.754003,0.470311


{'steps': 36, 'initial_train_size': '2012-08-31 23:59:00', 'refit': False, 'fixed_train_size': True, 'gap': 0, 'fold_stride': 36, 'differentiation': None}


╭───────────────────────── Skforecast-AI Results Explanation ──────────────────────────╮
│ Initial training up to 2012-08-31 23:59:00, fixed window, no refit, 36-step horizon, │
│ 82 folds. Results — mean_absolute_error: 46.5843, mean_squared_error: 5443.1711,     │
│ mean_absolute_scaled_error: 0.7540, mean_absolute_percentage_error: 0.4703.          │
╰──────────────────────────────────────────────────────────────────────────────────────╯

In [18]:
syntax = Syntax(
    result.code, 
    "python", 
    theme="monokai",
    word_wrap=True
)
console = Console()
console.print(syntax)

import pandas as pd                                                                                                
from lightgbm import LGBMRegressor                                                                                 
from skforecast.preprocessing import RollingFeatures, CalendarFeatures                                             
from skforecast.recursive import ForecasterRecursive                                                               
from skforecast.model_selection import TimeSeriesFold, backtesting_forecaster                                      
                                                                                                                   
# Load data                                                                                                        
data = pd.read_csv('data.csv', index_col=0, parse_dates=True)                                                      
                                                                                                                   
data = data.asfreq('h')                                                                                            
data = data.sort_index()                                                                                           
                                                                                                                   
window_features = RollingFeatures(                                                                                 
    stats        = ['mean', 'std', 'mean', 'mean'],                                                                
    window_sizes = [3, 3, 24, 168],                                                                                
)                                                                                                                  
                                                                                                                   
calendar_features = CalendarFeatures(                                                                              
    features = ['hour', 'day_of_week', 'weekend', 'month'],                                                        
    encoding = None,                                                                                               
)                                                                                                                  
                                                                                                                   
# Create forecaster                                                                                                
forecaster = ForecasterRecursive(                                                                                  
    estimator            = LGBMRegressor(random_state=123, verbose=-1),                                            
    lags                 = [1, 2, 3, 5, 8, 10, 15, 17, 19, 20, 21, 22, 23, 24, 25, 26, 32, 33, 119, 121, 135, 136, 
142, 143, 145, 160, 166, 167, 169, 313, 335, 337],                                                                 
    window_features      = window_features,                                                                        
    calendar_features    = calendar_features,                                                                      
    categorical_features [38;2;255;70;137;48